In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
customer_name = 'Cadent'
customer_id = Query(query = f"SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'").execute([KPIHub_Conn]).values[0][0]


tableList = [KPI_ReportSummary]
aggregator = ['BoundaryRegion','ReportYear', 'ReportWeek']
data = {}
for table in tableList:
    data[table] = Query(query = f"SELECT * FROM {table.name} WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'))").execute([KPIHub_Conn])

def report_summary_apply(df):
    return pd.Series({
        'FOVMain': df['DistributionPipeCoveredKm'].sum()/df['DistributionPipeKm'].sum(),
        'ReportAssetLengthKm': df['ReportAssetLengthKm'].sum() if 'ReportAssetLengthKm' in df else None,
        'AssetCoveredLengthKm': df['AssetCoveredLengthKm'].sum() if 'AssetCoveredLengthKm' in df else None,
        'DistributionPipeKm': df['DistributionPipeKm'].sum() if 'DistributionPipeKm' in df else None,
        'DistributionPipeCoveredKm': df['DistributionPipeCoveredKm'].sum() if 'DistributionPipeCoveredKm' in df else None,
        'ServicePipeKm': df['ServicePipeKm'].sum() if 'ServicePipeKm' in df else None,
        'ServicePipeCoveredKm': df['ServicePipeCoveredKm'].sum() if 'ServicePipeCoveredKm' in df else None,
        'ReportCount': df.shape[0]
    })

report_by_period = data[KPI_ReportSummary].groupby(aggregator).apply(report_summary_apply)
report_by_period = report_by_period.round(2)

/tmp/ipykernel_1108922/2416937470.py:13: RuntimeWarning: invalid value encountered in double_scalars
  'FOVMain': df['DistributionPipeCoveredKm'].sum()/df['DistributionPipeKm'].sum(),


In [4]:
report_by_period

FOVMain  ReportAssetLengthKm  \
BoundaryRegion ReportYear ReportWeek                                 
East Midlands  2024       16              NaN               138.45   
                          35             0.95                96.41   
                          36             0.96                91.50   
                          48             0.94                72.62   
               2026       5              0.97               129.55   
...                                       ...                  ...   
West Midlands  2026       23             0.94               836.49   
                          24             0.94               791.94   
                          25             0.94               962.79   
                          26             0.96               404.50   
                          27             0.95               262.75   

                                      AssetCoveredLengthKm  \
BoundaryRegion ReportYear ReportWeek                         
East Midlands  2024       16                        129.60   
                          35                         90.95   
                          36                         88.09   
                          48                         67.69   
               2026       5                         124.92   
...                                                    ...   
West Midlands  2026       23                        785.09   
                          24                        739.49   
                          25                        901.44   
                          26                        386.09   
                          27                        247.82   

                                      DistributionPipeKm  \
BoundaryRegion ReportYear ReportWeek                       
East Midlands  2024       16                        0.00   
                          35                       94.34   
                          36                       89.74   
                          48                       71.22   
               2026       5                       126.68   
...                                                  ...   
West Midlands  2026       23                      821.33   
                          24                      772.27   
                          25                      937.96   
                          26                      397.23   
                          27                      257.53   

                                      DistributionPipeCoveredKm  \
BoundaryRegion ReportYear ReportWeek                              
East Midlands  2024       16                               0.00   
                          35                              89.30   
                          36                              86.59   
                          48                              66.79   
               2026       5                              122.30   
...                                                         ...   
West Midlands  2026       23                             773.17   
                          24                             724.74   
                          25                             883.48   
                          26                             380.35   
                          27                             243.82   

                                      ServicePipeKm  ServicePipeCoveredKm  \
BoundaryRegion ReportYear ReportWeek                                        
East Midlands  2024       16                   0.00                  0.00   
                          35                   2.07                  1.65   
                          36                   1.76                  1.50   
                          48                   1.40                  0.90   
               2026       5                    2.88                  2.62   
...                                             ...                   ...   
West Midlands  2026       23                  15.16

In [5]:
r = report_by_period.reset_index()
r_long = r.melt(id_vars=aggregator, var_name='KPIId', value_name='Value')
r_long['Id'] = r_long.apply(lambda row: f"{row['KPIId']}_{customer_name}_R{row['BoundaryRegion'].replace(' ','')}_Y{row['ReportYear']}_W{row['ReportWeek']}", axis=1)
r_long = r_long.rename(columns={'ReportWeek': 'PeriodValue', 'ReportYear': 'Year'})
r_long['PeriodType'] = "Weekly"
r_long['LastUpdated'] = datetime.now()
r_long['CustomerId'] = customer_id

In [6]:
r_long

,BoundaryRegion,Year,PeriodValue,KPIId,Value,Id,PeriodType,LastUpdated,CustomerId
0,East Midlands,2024,16,FOVMain,NaN,FOVMain_Cadent_REastMidlands_Y2024_W16,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
1,East Midlands,2024,35,FOVMain,0.95,FOVMain_Cadent_REastMidlands_Y2024_W35,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
2,East Midlands,2024,36,FOVMain,0.96,FOVMain_Cadent_REastMidlands_Y2024_W36,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
3,East Midlands,2024,48,FOVMain,0.94,FOVMain_Cadent_REastMidlands_Y2024_W48,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
4,East Midlands,2026,5,FOVMain,0.97,FOVMain_Cadent_REastMidlands_Y2026_W5,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
...,...,...,...,...,...,...,...,...,...
2211,West Midlands,2026,23,ReportCount,25.00,ReportCount_Cadent_RWestMidlands_Y2026_W23,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
2212,West Midlands,2026,24,ReportCount,25.00,ReportCount_Cadent_RWestMidlands_Y2026_W24,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
2213,West Midlands,2026,25,ReportCount,28.00,ReportCount_Cadent_RWestMidlands_Y2026_W25,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98
2214,West Midlands,2026,26,ReportCount,12.00,ReportCount_Cadent_RWestMidlands_Y2026_W26,Weekly,2026-06-30 12:37:35.484961,BD4D080B-1D12-D329-ABD0-39FEB9804E98


In [7]:
KPI_Data.update_table(arguments = {'DataFrame': r_long, 'db_path': DB_PATH, 'PrimaryKey': 'Id'})

In [8]:
KPI_Data.query_table(arguments = {'db_path': DB_PATH})

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_REastMidlands_Y2024_W16,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,16,None,None,2026-06-30 12:37:35.484961
1,FOVMain_Cadent_REastMidlands_Y2024_W35,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,35,0.95,None,2026-06-30 12:37:35.484961
2,FOVMain_Cadent_REastMidlands_Y2024_W36,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,36,0.96,None,2026-06-30 12:37:35.484961
3,FOVMain_Cadent_REastMidlands_Y2024_W48,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,48,0.94,None,2026-06-30 12:37:35.484961
4,FOVMain_Cadent_REastMidlands_Y2026_W5,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2026,Weekly,5,0.97,None,2026-06-30 12:37:35.484961
...,...,...,...,...,...,...,...,...,...,...
21339,DistributionPipeKm_Cadent_RNorthLondon_Y2026_W27,DistributionPipeKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,71.06,None,2026-06-30 12:37:35.484961
21340,DistributionPipeCoveredKm_Cadent_RNorthLondon_...,DistributionPipeCoveredKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,65.01,None,2026-06-30 12:37:35.484961
21341,ServicePipeKm_Cadent_RNorthLondon_Y2026_W27,ServicePipeKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,2.91,None,2026-06-30 12:37:35.484961
21342,ServicePipeCoveredKm_Cadent_RNorthLondon_Y2026...,ServicePipeCoveredKm,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,2.59,None,2026-06-30 12:37:35.484961
